# T9: Map visualization of T2 and T3 results
*Co-authored with CoCo*

**T2 (data quality):** quality issue percentage per taxi zone for all 4 datasets  
**T3 (exploratory analysis):** trip volume per taxi zone for all 4 datasets

In [ ]:
%%sql -r wh
USE WAREHOUSE BIGDATA_MZMB_WH

In [ ]:
import pandas as pd
import numpy as np
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

quality = session.sql('SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.T9_QUALITY_BY_ZONE').to_pandas()
volume = session.sql('SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.T3_PICKUP_LOCATION_DIST').to_pandas()
dropoff = session.sql('SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.T3_TOP_DROPOFF_LOCATIONS').to_pandas()
od_pairs = session.sql('SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.T3_TOP_OD_PAIRS').to_pandas()
geom = session.sql('SELECT LOCATION_ID, BOROUGH, ZONE, ST_ASGEOJSON(GEOM) AS GEOJSON, ST_X(ST_CENTROID(GEOM)) AS LON, ST_Y(ST_CENTROID(GEOM)) AS LAT FROM BIGDATA_TAXI_MZMB.EXTERNAL_DATA.TAXI_ZONES_GEOM').to_pandas()

print(f"T2 quality by zone: {len(quality)} rows")
print(f"T3 pickup volume: {len(volume)} rows")
print(f"T3 dropoff top locations: {len(dropoff)} rows")
print(f"T3 OD pairs: {len(od_pairs)} rows")
print(f"Zone geometries: {len(geom)} zones")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
import matplotlib.colors as mcolors

def parse_geojson_polygons(geojson_str):
    g = json.loads(geojson_str)
    polys = []
    if g['type'] == 'Polygon':
        polys.append(g['coordinates'][0])
    elif g['type'] == 'MultiPolygon':
        for poly in g['coordinates']:
            polys.append(poly[0])
    return polys

zone_polys = {}
for _, row in geom.iterrows():
    try:
        polys = parse_geojson_polygons(row['GEOJSON'])
        zone_polys[int(row['LOCATION_ID'])] = polys
    except:
        pass

print(f"Parsed polygons for {len(zone_polys)} zones")

In [ ]:
def draw_zone_map(ax, data_df, colormap, norm, id_col='LOCATION_ID', val_col='TOTAL_ISSUE_PCT'):
    data_df = data_df.copy()
    data_df[id_col] = data_df[id_col].astype(int)
    for _, row in data_df.iterrows():
        loc_id = row[id_col]
        if loc_id not in zone_polys:
            continue
        val = min(row[val_col], norm.vmax)
        color = colormap(norm(val))
        for coords in zone_polys[loc_id]:
            poly = MplPolygon(coords, closed=True, facecolor=color, edgecolor='black', linewidth=0.2, alpha=0.8)
            ax.add_patch(poly)
    for loc_id, polys in zone_polys.items():
        if loc_id not in data_df[id_col].values:
            for coords in polys:
                poly = MplPolygon(coords, closed=True, facecolor='#f0f0f0', edgecolor='gray', linewidth=0.15, alpha=0.4)
                ax.add_patch(poly)
    ax.set_xlim(-74.27, -73.7)
    ax.set_ylim(40.49, 40.92)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

datasets = ['yellow', 'green', 'fhv', 'fhvhv']
titles = ['Yellow taxi', 'Green taxi', 'FHV', 'FHVHV (Uber/Lyft)']

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
vmax = quality['TOTAL_ISSUE_PCT'].quantile(0.95)
norm = mcolors.Normalize(vmin=0, vmax=vmax)
cmap = plt.colormaps['YlOrRd']

for ax, ds, title in zip(axes.flatten(), datasets, titles):
    subset = quality[quality['DATASET'] == ds]
    draw_zone_map(ax, subset, cmap, norm)
    ax.set_title(title, fontsize=11)

fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Data quality issues (% of trips)', fontsize=10)
plt.suptitle('T2 data quality issues by taxi zone', fontsize=13, y=0.95)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
vmax = volume['TRIP_COUNT'].quantile(0.95)
norm = mcolors.LogNorm(vmin=max(volume['TRIP_COUNT'].min(), 1), vmax=vmax)
cmap = plt.colormaps['Blues']

for ax, ds, title in zip(axes.flatten(), datasets, titles):
    subset = volume[volume['DATASET'] == ds].rename(columns={'PU_LOCATION_ID': 'LOCATION_ID'})
    draw_zone_map(ax, subset, cmap, norm, val_col='TRIP_COUNT')
    ax.set_title(title, fontsize=11)

fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Trip count (log scale)', fontsize=10)
plt.suptitle('Trip volume by taxi zone (from T3)', fontsize=13, y=0.95)
plt.show()

In [ ]:
# T3: Top OD pair flows (lines between zone centroids)
centroids = geom.set_index('LOCATION_ID')[['LON', 'LAT']].to_dict('index')

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

for ax, ds, title in zip(axes.flatten(), datasets, titles):
    # Draw base zones in light gray
    for loc_id, polys in zone_polys.items():
        for coords in polys:
            poly = MplPolygon(coords, closed=True, facecolor='#f5f5f5', edgecolor='gray', linewidth=0.15, alpha=0.6)
            ax.add_patch(poly)
    
    # Draw OD flow lines
    subset = od_pairs[od_pairs['DATASET'] == ds].sort_values('TRIP_COUNT', ascending=False).head(15)
    max_trips = subset['TRIP_COUNT'].max()
    
    for _, row in subset.iterrows():
        pu = int(row['PU_LOCATION_ID'])
        do = int(row['DO_LOCATION_ID'])
        if pu not in centroids or do not in centroids:
            continue
        x1, y1 = centroids[pu]['LON'], centroids[pu]['LAT']
        x2, y2 = centroids[do]['LON'], centroids[do]['LAT']
        lw = 0.5 + 4 * (row['TRIP_COUNT'] / max_trips)
        alpha = 0.4 + 0.5 * (row['TRIP_COUNT'] / max_trips)
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                   arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=lw, alpha=alpha))
    
    ax.set_xlim(-74.27, -73.7)
    ax.set_ylim(40.49, 40.92)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'{title} (top 15 OD pairs)', fontsize=11)

plt.suptitle('T3: Top origin-destination flows', fontsize=13, y=0.95)
plt.tight_layout()
plt.show()

In [ ]:
# Load additional T3 temporal/spatial data
covid_zone = session.sql('SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.T9_COVID_BY_ZONE').to_pandas()
hourly_zone = session.sql('SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.T9_HOURLY_BY_ZONE').to_pandas()
recovery_zone = session.sql('SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.T9_RECOVERY_BY_ZONE').to_pandas()
print(f"COVID impact: {len(covid_zone)} zones")
print(f"Hourly patterns: {len(hourly_zone)} zones")
print(f"Recovery (2024 vs 2019): {len(recovery_zone)} zones")

In [ ]:
# T3: COVID-19 impact by zone (% change 2019 vs 2020)
fig, ax = plt.subplots(figsize=(12, 10))
covid_zone_clean = covid_zone[covid_zone['PCT_CHANGE'].notna()].copy()
norm = mcolors.TwoSlopeNorm(vmin=-100, vcenter=-70, vmax=0)
cmap_div = plt.colormaps['RdYlGn']

draw_zone_map(ax, covid_zone_clean, cmap_div, norm, val_col='PCT_CHANGE')
ax.set_title('COVID-19 impact: yellow taxi trip change 2020 vs 2019 (%)\n(green = less decline, red = severe decline)', fontsize=11)

sm = plt.cm.ScalarMappable(cmap=cmap_div, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('% change (2020 vs 2019)', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# T3: Peak hour by zone
fig, ax = plt.subplots(figsize=(12, 10))
norm = mcolors.Normalize(vmin=0, vmax=23)
cmap_hour = plt.colormaps['twilight_shifted']

draw_zone_map(ax, hourly_zone, cmap_hour, norm, val_col='PEAK_HOUR')
ax.set_title('Peak pickup hour by zone (yellow taxi)', fontsize=11)

sm = plt.cm.ScalarMappable(cmap=cmap_hour, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Peak hour (0-23)', fontsize=9)
cbar.set_ticks([0, 6, 12, 18, 23])
cbar.set_ticklabels(['midnight', '6am', 'noon', '6pm', '11pm'])
plt.tight_layout()
plt.show()

In [ ]:
# T3: Post-COVID recovery (2024 vs 2019)
fig, ax = plt.subplots(figsize=(12, 10))
recovery_clean = recovery_zone[recovery_zone['RECOVERY_PCT'].notna()].copy()
recovery_clean['RECOVERY_PCT'] = recovery_clean['RECOVERY_PCT'].clip(0, 150)
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=50, vmax=150)
cmap_rec = plt.colormaps['RdYlGn']

draw_zone_map(ax, recovery_clean, cmap_rec, norm, val_col='RECOVERY_PCT')
ax.set_title('Post-COVID recovery: 2024 avg monthly trips as % of 2019\n(green = fully recovered, red = still below pre-COVID)', fontsize=11)

sm = plt.cm.ScalarMappable(cmap=cmap_rec, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Recovery % (100% = back to 2019 level)', fontsize=9)
plt.tight_layout()
plt.show()